# Proyecto Final Parte 2 - Reproductor MIDI

### Nombre: Roberto Alejandro Chagra Martinez
### Codigo: 208447128

In [1]:
#@title Preparacion: Instalar sintetizador MIDI

# Instalar el sintetizador MIDI y su banco de sonidos
!apt-get -qq install -y fluidsynth fluid-soundfont-gm

Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1build1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../04-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.9build

In [2]:
#@title Preparacion: Instalación bibliotecas

# Instalar las bibliotecas de Python
!pip -q install kagglehub pretty_midi ipywidgets soundfile matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 54.3 MB/s eta 0:00:00


In [3]:
#@title Preparacion: Importación de librerias

import os, json, html, base64, tempfile, subprocess
import io
from pathlib import Path
from google.colab import files, output
import numpy as np
import soundfile as sf
import pretty_midi
import ipywidgets as widgets
from IPython.display import display, Audio, HTML, clear_output

In [6]:
#@title 0. Dataset

# Se define la ruta de la carpeta en Colab
ruta_carpeta = Path('/content/MIDIS') if Path('/content/MIDIS').exists() else Path('examples')

# Buscamos automáticamente todos los archivos con extensión .mid o .midi dentro de esa carpeta
archivos_midi = sorted(list(ruta_carpeta.glob('*.mid')))

# Definir la variable 'datasets' mapeada para la interfaz
datasets = {
    'Mis Canciones': archivos_midi
}

In [7]:
#@title 1. Iterfaz

# Banco de sonidos MIDI
SOUNDFONT = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

# Crear el reproductor HTML con controles y piano roll
def player_page(wav_bytes, notes, title):
    encoded = base64.b64encode(wav_bytes).decode()

    page = r'''<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
body {
    margin: 0;
    padding: 14px;
    background: #101724;
    color: #eaf1ff;
    font: 14px system-ui;
}
.panel {
    padding: 17px;
    background: #182235;
    border: 1px solid #344052;
    border-radius: 16px;
}
h3 {
    margin: 0 0 14px;
    font-size: 17px;
}
.controls {
    display: flex;
    gap: 10px;
    align-items: center;
    flex-wrap: wrap;
}
button {
    background: #263e61;
    color: white;
    border: 1px solid #50658a;
    border-radius: 10px;
    padding: 10px;
    cursor: pointer;
}
button:hover {
    background: #365b88;
}
#seek {
    flex: 1;
    min-width: 140px;
    accent-color: #60b5ff;
}
canvas {
    display: block;
    background: #0e1524;
    width: 100%;
    height: 165px;
    border-radius: 9px;
    margin-top: 14px;
}
.sub {
    color: #aabcce;
    font-size: 12px;
    margin-top: 14px;
}
#msg {
    color: #9ec7ea;
    font-size: 12px;
    margin-top: 10px;
}
</style>
</head>

<body>
<div class="panel">
    <h3 id="title"></h3>

    <div class="controls">
        <button id="play">▶ Reproducir</button>
        <button id="reset">↺ Inicio</button>
        <span id="time">0:00 / 0:00</span>
        <input id="seek" type="range" min="0" max="1000" value="0">
    </div>

    <div class="sub">
        Piano Roll MIDI · Línea celeste = posición de reproducción
    </div>

    <canvas id="roll"></canvas>
    <div id="msg">Preparando audio...</div>
</div>

<script>
const WAV = '__WAV__';
const NOTES = __NOTES__;
const TITLE = __TITLE__;

const $ = id => document.getElementById(id);
$('title').textContent = TITLE;

let ctx, buffer, source = null;
let started = 0, offset = 0, playing = false, duration = 0;

const canvas = $('roll');
const g = canvas.getContext('2d');

// Formatear segundos a minutos
const fmt = t => {
    t = Math.max(0, Math.floor(t));
    return Math.floor(t / 60) + ':' + String(t % 60).padStart(2, '0');
};

// Obtener la posición actual
const position = () =>
    playing ? Math.min(duration, offset + ctx.currentTime - started) : offset;

// Actualizar tiempo y barra de progreso
function update() {
    const p = position();
    $('time').textContent = fmt(p) + ' / ' + fmt(duration);
    $('seek').value = duration ? Math.round(p / duration * 1000) : 0;
}

// Cargar el audio
async function init() {
    try {
        ctx = new (window.AudioContext || window.webkitAudioContext)();

        const bin = atob(WAV);
        const bytes = new Uint8Array(bin.length);

        for (let i = 0; i < bin.length; i++)
            bytes[i] = bin.charCodeAt(i);

        buffer = await ctx.decodeAudioData(bytes.buffer.slice(0));
        duration = buffer.duration;

        $('msg').textContent = 'Listo para reproducir';
        update();

    } catch (e) {
        $('msg').textContent = 'Error: ' + e.message;
    }
}

// Pausar reproducción
function pause(keep = true) {
    if (playing) {
        let p = position();

        source.onended = null;
        try { source.stop(); } catch (e) {}

        source.disconnect();
        playing = false;
        source = null;

        if (keep) offset = p;
    }

    $('play').textContent = '▶ Reproducir';
    update();
}

// Reproducir audio
async function play() {
    if (!buffer) return;

    await ctx.resume();

    if (playing) {
        pause();
        return;
    }

    if (offset >= duration - 0.01) offset = 0;

    source = ctx.createBufferSource();
    source.buffer = buffer;
    source.connect(ctx.destination);

    started = ctx.currentTime;

    source.onended = () => {
        if (playing) {
            pause(false);
            offset = duration;
            update();
        }
    };

    source.start(0, offset);
    playing = true;
    $('play').textContent = '⏸ Pausar';
}

// Botones de reproducción
$('play').onclick = play;

$('reset').onclick = () => {
    let was = playing;
    pause();
    offset = 0;
    was ? play() : update();
};

// Adelantar o retroceder canción
$('seek').oninput = e => {
    let was = playing;
    pause();
    offset = Number(e.target.value) / 1000 * duration;
    was ? play() : update();
};

// Dibujar notas musicales en tiempo real
function draw() {
    const dpr = window.devicePixelRatio || 1;
    const w = canvas.clientWidth;
    const h = canvas.clientHeight;

    if (canvas.width !== Math.round(w * dpr) ||
        canvas.height !== Math.round(h * dpr)) {
        canvas.width = Math.round(w * dpr);
        canvas.height = Math.round(h * dpr);
    }

    g.setTransform(dpr, 0, 0, dpr, 0, 0);
    g.clearRect(0, 0, w, h);

    const pos = position();
    const from = Math.max(0, pos - 2);
    const to = from + 12;

    // Fondo del piano roll
    for (let i = 0; i < 12; i++) {
        g.fillStyle = i % 2 ? '#111d30' : '#152238';
        g.fillRect(0, i * h / 12, w, h / 12);
    }

    // Dibujar notas MIDI
    for (const [s, e, p, v] of NOTES) {
        if (e < from || s > to) continue;

        g.fillStyle = v > 90 ? '#55c5ee' : '#6998ff';

        g.fillRect(
            (s - from) * w / 12,
            (96 - p) * h / 60,
            Math.max(2, (e - s) * w / 12),
            Math.max(2, h / 60)
        );
    }

    // Línea de reproducción
    g.strokeStyle = '#94f9f0';
    g.lineWidth = 2;
    g.beginPath();
    g.moveTo((pos - from) * w / 12, 0);
    g.lineTo((pos - from) * w / 12, h);
    g.stroke();

    update();
    requestAnimationFrame(draw);
}

init();
draw();
</script>
</body>
</html>'''

    page = page.replace('__WAV__', encoded)
    page = page.replace('__NOTES__', json.dumps(notes))
    page = page.replace('__TITLE__', json.dumps(title))

    return f'<iframe style="width:100%;height:350px;border:0" allow="autoplay" srcdoc="{html.escape(page, quote=True)}"></iframe>'


# ========================================
# CONTROLES DE LA INTERFAZ
# ========================================

# Seleccionar dataset
dataset_select = widgets.Dropdown(
    options=list(datasets),
    description='Dataset:'
)

# Seleccionar canción
song_select = widgets.Dropdown(
    description='Canción:',
    layout=widgets.Layout(width='95%')
)

# Seleccionar instrumento MIDI
inst_select = widgets.Dropdown(
    options=[
        (pretty_midi.program_to_instrument_name(i), i)
        for i in range(128)
    ],
    description='Instrumento:',
    layout=widgets.Layout(width='95%')
)

# Conservar instrumentos originales
original = widgets.Checkbox(
    value=False,
    description='Conservar instrumentos originales'
)

# Botón para cargar la canción
load = widgets.Button(
    description='🎵 Cargar canción',
    button_style='primary'
)

out = widgets.Output()


# Actualizar canciones al cambiar el dataset
def actualizar(change=None):
    song_select.options = [
        (p.name, i)
        for i, p in enumerate(datasets[dataset_select.value])
    ]


# ========================================
# CARGAR Y REPRODUCIR MIDI
# ========================================

def cargar(_):
    with out:
        clear_output(wait=True)

        if song_select.value is None:
            print('No hay canciones disponibles')
            return

        archivo = datasets[dataset_select.value][song_select.value]

        print('Generando audio:', archivo.name)

        try:
            # Leer canción MIDI
            midi = pretty_midi.PrettyMIDI(str(archivo))

            # Cambiar instrumento si no se usa el original
            if not original.value:
                for pista in midi.instruments:
                    if not pista.is_drum:
                        pista.program = inst_select.value

            # Obtener notas para el piano roll
            notes = sorted(
                (
                    round(n.start, 3),
                    round(n.end, 3),
                    n.pitch,
                    n.velocity
                )
                for pista in midi.instruments
                for n in pista.notes
                if 36 <= n.pitch <= 96
            )[:35000]

            # Convertir MIDI a WAV
            with tempfile.TemporaryDirectory() as tmp:
                midi_file = f'{tmp}/song.mid'
                wav_file = f'{tmp}/song.wav'

                midi.write(midi_file)

                subprocess.run([
                    'fluidsynth', '-ni',
                    '-F', wav_file,
                    '-r', '16000',
                    '-g', '0.35',
                    SOUNDFONT,
                    midi_file
                ], check=True, capture_output=True)

                # Convertir audio estéreo a mono
                audio, sr = sf.read(wav_file, always_2d=True)
                sf.write(
                    wav_file,
                    audio.mean(axis=1),
                    sr,
                    subtype='PCM_16'
                )

                wav = Path(wav_file).read_bytes()

            # Evitar archivos demasiado grandes
            if len(wav) > 19_000_000:
                print('Audio demasiado largo. Elige una canción más corta.')
                return

            # Mostrar reproductor
            clear_output(wait=True)

            display(HTML(player_page(wav, notes, archivo.name)))

            print(
                f'{archivo.name} · '
                f'{len(midi.instruments)} pistas · '
                f'{midi.get_end_time():.1f} segundos'
            )

        except Exception as e:
            print('Error:', e)


# ========================================
# MOSTRAR LA INTERFAZ
# ========================================

dataset_select.observe(actualizar, names='value')
load.on_click(cargar)
actualizar()

display(widgets.VBox([
    widgets.HTML('<h2>🎵 MIDI Studio</h2>'),
    dataset_select,
    song_select,
    inst_select,
    original,
    load,
    out
]))

In [8]:
#@title 2. Carga de MIDI

output.enable_custom_widget_manager()

# 1. Subir archivos MIDI
print("📁 Selecciona tus archivos MIDI")
archivos = files.upload()

# 2. Crear interfaz
cancion = widgets.Dropdown(
    options=list(archivos),
    description="Canción:",
    layout=widgets.Layout(width="95%")
)

instrumento = widgets.Dropdown(
    options=[("Original", -1)] +
    [(pretty_midi.program_to_instrument_name(i), i)
     for i in range(128)],
    description="Instrumento:",
    layout=widgets.Layout(width="95%")
)

boton = widgets.Button(
    description="▶ Reproducir MIDI",
    button_style="primary",
    layout=widgets.Layout(width="220px")
)

pantalla = widgets.Output()

# 3. Reproducir la canción seleccionada
def reproducir(b):
    with pantalla:
        clear_output(wait=True)

        if not cancion.value:
            print("Primero sube un MIDI")
            return

        try:
            midi = pretty_midi.PrettyMIDI(
                io.BytesIO(archivos[cancion.value])
            )

            if instrumento.value != -1:
                for pista in midi.instruments:
                    if not pista.is_drum:
                        pista.program = instrumento.value

            print("🎵 Preparando:", cancion.value)

            with tempfile.TemporaryDirectory() as carpeta:
                mid = str(Path(carpeta) / "cancion.mid")
                wav = str(Path(carpeta) / "cancion.wav")

                midi.write(mid)

                subprocess.run([
                    "fluidsynth", "-ni",
                    "-F", wav, "-r", "16000",
                    "/usr/share/sounds/sf2/FluidR3_GM.sf2",
                    mid
                ], check=True, capture_output=True)

                clear_output(wait=True)
                print("🎵", cancion.value)
                print(f"Duración: {midi.get_end_time():.1f} segundos")
                display(Audio(filename=wav))

        except Exception as e:
            print("❌ Error:", e)

boton.on_click(reproducir)

📁 Selecciona tus archivos MIDI


Saving Titanic.mid to Titanic.mid


In [9]:
#@title 3. Mostrar Reproductor

display(widgets.VBox([
    widgets.HTML("<h2>🎹 MIDI Studio</h2>"),
    widgets.HTML("<b>Mis archivos MIDI</b>"),
    cancion,
    instrumento,
    boton,
    pantalla
], layout=widgets.Layout(
    border="1px solid #50658a",
    padding="18px",
    width="96%"
)))